# 03. Models

As always start by loading the data and preparing it appropriately.

In [2]:
import sys
sys.path.append('..')

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_validate

from src.dataset import load_clients, add_year_block, time_split
from src.features import TARGET, prepare
from src.models import build_pipeline

df = add_year_block(load_clients())
train, test = time_split(df)
X_train, y_train = prepare(train), train[TARGET]

We split the test data into five different folds.
Fold 1 means train on chunk 1 and test on chunk 2, fold 2 means train on chunks 1-2 and test on 3, fold 3 means train on 1-3 and test on 4, and so on.
The chunks are in chronological order, so we are never using the future to predict the past.

In [8]:
cv = TimeSeriesSplit(n_splits=5)
for i, (tr, te) in enumerate(cv.split(X_train), start=1):
    print(i, len(tr), len(te), y_train.iloc[te].mean().round(3))  # First look at the folds

1 5490 5490 0.039
2 10980 5490 0.06
3 16470 5490 0.059
4 21960 5490 0.054
5 27450 5490 0.14


We now look at the ROC AUC and PR AUC scores after each fold.

In [25]:
scoring = {"ROC_AUC": "roc_auc", "PR_AUC": "average_precision"}

logreg = build_pipeline(LogisticRegression(max_iter=1000), include_macro=True)
res = cross_validate(logreg, X_train, y_train, cv=cv, scoring=scoring)
tb = pd.DataFrame(res)[["test_ROC_AUC", "test_PR_AUC"]].round(3)
tb["base_rate"] = [y_train.iloc[te].mean().round(3) for _, te in cv.split(X_train)]
tb["PR_AUC / base"] = round(tb["test_PR_AUC"] / tb["base_rate"], 3)
tb

,test_ROC_AUC,test_PR_AUC,base_rate,PR_AUC / base
0,0.529,0.040,0.039,1.026
1,0.566,0.073,0.060,1.217
2,0.496,0.061,0.059,1.034
3,0.447,0.051,0.054,0.944
4,0.638,0.199,0.140,1.421


Folds 1-4 show essentially no skill, only after fold 4 can we talk about that.
As the increase actually goes down between the 2nd and 4th folds, it could be that in the beginning patterns were learned that were later reversed.
The average ROC AUC score is about 0.54, not much better than random guessing.

To diagnose this a bit, let us look at the ROC AUC score when we use the trained model back on the trained data.

In [28]:
from sklearn.metrics import roc_auc_score

logreg.fit(X_train, y_train)
print(roc_auc_score(y_train, logreg.predict_proba(X_train)[:, 1]))

0.6824119881415601


A ROC AUC score of 0.68 is not bad (it's above the 0.5 threshold), but still modest.
It could be that a linear model is not enough here because the link between features and subscribing differs from period to period.
We will later see if gradient boosting fixes this.

Another possibility comes from the fact that in each fold we use the trained model on months that didn't appear in the training.
This may be unfair because in the final test we will have trained on all months.
We can check how many unseen months there are in the test data of each fold:

In [29]:
for i, (tr, te) in enumerate(cv.split(X_train), start=1):
    seen = set(train.iloc[tr]["month"])
    unseen = (~train.iloc[te]["month"].isin(seen)).mean().round(2)
    print(i, "share of scoring rows with a month not seen in training:", unseen)

1 share of scoring rows with a month not seen in training: 0.59
2 share of scoring rows with a month not seen in training: 0.79
3 share of scoring rows with a month not seen in training: 0.57
4 share of scoring rows with a month not seen in training: 0.63
5 share of scoring rows with a month not seen in training: 0.5


In all cases, more than 50% of the scoring rows come from months that were not seen in training.
This means that the training folds are actually harder than the actual test, at least in this specific way.

Finally, let us try a shuffled cross-validation.
We train on the same data but shuffle the rows, i.e., we mix all time periods together.

In [30]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
res_random = cross_validate(logreg, X_train, y_train, cv=skf, scoring=scoring)
print(res_random["test_ROC_AUC"].round(3), res_random["test_ROC_AUC"].mean().round(3))
print(res_random["test_PR_AUC"].round(3), "base rate:", y_train.mean().round(3))

[0.689 0.662 0.67  0.67  0.663] 0.671
[0.212 0.226 0.196 0.19  0.199] base rate: 0.064


We get a score of 0.67, almost the same as what we got when scoring on the training data a few cells above.
This means that, when time is not being taken into account, the model does as well on data it hasn't seen as on data it has seen.
However, once we train on a period and test on the following one, the score drops to 0.54 on average.

When we mix periods, the model learns in part which periods are good (i.e., the macro conditions and months) and in part something about the actual clients. To separate the two, let us run the model with shuffled rows but this time excluding the macros:

In [35]:
logreg_nm = build_pipeline(LogisticRegression(max_iter=1000), include_macro=False)
r = cross_validate(logreg_nm, X_train, y_train, cv=skf, scoring=scoring)
print(f'Shuffled, no macros: ROC AUC {r["test_ROC_AUC"].mean():.3f}, PR AUC {r["test_PR_AUC"].mean():.3f}')

Shuffled, no macros: ROC AUC 0.666, PR AUC 0.204


The ROC AUC score barely fell from the 0.67-0.68 marks above.
This excludes the possibility that much of the learned information was related to the time period.
Instead, it's telling us that the skill does come from client and contact features, but these features change over time.

In [38]:
def cv_table(model):
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    table = pd.DataFrame(res)[["test_ROC_AUC", "test_PR_AUC"]]
    table["base_rate"] = [y_train.iloc[te].mean() for _, te in cv.split(X_train)]
    table["PR_AUC / base"] = table["test_PR_AUC"] / table["base_rate"]
    table.index = range(1, len(table) + 1)
    table.index.name = "fold"
    return table.round(3)

from sklearn.ensemble import HistGradientBoostingClassifier

boost = HistGradientBoostingClassifier(early_stopping=False, random_state=0)

candidates = {
    "logreg + macro": build_pipeline(LogisticRegression(max_iter=1000), include_macro=True),
    "logreg, no macro": build_pipeline(LogisticRegression(max_iter=1000), include_macro=False),
    "boosting + macro": build_pipeline(boost, include_macro=True),
    "boosting, no macro": build_pipeline(boost, include_macro=False),
}

tables = {label: cv_table(mdl) for label, mdl in candidates.items()}

In [39]:
pd.DataFrame({label: t["test_ROC_AUC"] for label, t in tables.items()})

,logreg + macro,"logreg, no macro",boosting + macro,"boosting, no macro"
fold,,,,
1,0.529,0.527,0.512,0.497
2,0.566,0.567,0.506,0.514
3,0.496,0.501,0.484,0.489
4,0.447,0.456,0.441,0.441
5,0.638,0.660,0.617,0.588


In [40]:
pd.DataFrame({label: t["PR_AUC / base"] for label, t in tables.items()})

,logreg + macro,"logreg, no macro",boosting + macro,"boosting, no macro"
fold,,,,
1,1.024,1.102,1.094,1.079
2,1.205,1.203,1.012,1.062
3,1.034,1.013,0.952,1.022
4,0.943,0.934,0.865,0.926
5,1.418,1.721,1.335,1.243
